# Tái lập The Scene Language trên Colab

Chạy lần lượt từng ô.

## 1. Lấy mã nguồn và cài đặt

In [ ]:
%cd /content
!git clone -q https://github.com/zzyunzhi/scene-language.git
!git clone -q https://github.com/LuuNhi1311/scene_language_experiment.git repro
!pip install -q -e /content/scene-language -r /content/repro/requirements.txt
%cd /content/scene-language
!git apply /content/repro/patches/scene-language.patch
%cd /content/repro

## 2. Khóa API

Tạo tại `platform.claude.com/settings/keys`. Không commit khóa lên GitHub.

In [ ]:
from getpass import getpass
key = getpass("ANTHROPIC_API_KEY: ")
open("/content/scene-language/engine/key.py", "w").write(f"ANTHROPIC_API_KEY = {key!r}\n")

## 3. Kiểm tra cài đặt và phần cứng

In [ ]:
!cd /content/scene-language && python scripts/tests/test_basic.py 2>&1 | tail -3
!grep -m1 'model name' /proc/cpuinfo; free -h | head -2; nvidia-smi -L || echo 'Không có GPU, nhánh Mitsuba vẫn chạy được'

## 4. Chạy thí nghiệm

Xem trước lệnh rồi mới chạy thật. Khoảng 10 phút.

In [ ]:
!python run_experiments.py --repo /content/scene-language --dry-run

In [ ]:
!python run_experiments.py --repo /content/scene-language

## 5. Lập bảng

Cột `count_correct` lấy từ `results/manual_grading.csv`. Nếu sinh mẫu mới thì phải chấm lại tay.

In [ ]:
!python analyze.py --repo /content/scene-language
import pandas as pd
pd.read_csv("results/results.csv").drop(columns="sample_path")

## 6. Đẩy lên Weights and Biases

In [ ]:
import wandb; wandb.login()
!python analyze.py --repo /content/scene-language --wandb

## 7. Tải kết quả về

In [ ]:
from google.colab import files
!zip -qr /content/results.zip results /content/scene-language/scripts/outputs
files.download("/content/results.zip")